In [75]:
import os
import warnings

from dotenv import load_dotenv
from langchain.chains import ConversationChain, RetrievalQA
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.memory import ChatMessageHistory, ConversationBufferMemory
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import CommaSeparatedListOutputParser, JsonOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [3]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"


def warn(*args, **kwargs):
    pass


warnings.warn = warn
warnings.filterwarnings("ignore")

os.environ["ANONYMIZED_TELEMETRY"] = "False"

load_dotenv()

True

In [50]:
model_id = "gpt-4o-mini"

parameters = {
    "max_tokens": 1000,
    "temperature": 0.7,
    "api_key": os.environ.get("OPENAI_API_KEY"),
    # "top_p": 1,
    # "frequency_penalty": 0,
    # "presence_penalty": 0
}

model = ChatOpenAI(model=model_id, **parameters)

# CHAT MODEL

The chat model takes a list of messages as input and returns a new message. All messages have both a role and a content property.

In [5]:
msg = model.invoke("Hello, how are you?")
print(msg.content)

Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


## CHAT MESSAGE

In [6]:
chat_messages = model.invoke(
    [
        SystemMessage(
            content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"
        ),
        HumanMessage(content="recommend a book about dogs"),
    ]
)

chat_messages.content

'I recommend "The Art of Racing in the Rain" by Garth Stein for a heartfelt perspective on life through a dog\'s eyes.'

# PROMPT TEMPLATES

Prompt templates help translate user input and parameters into instructions for a language model. 

In [7]:
input_variables = ["adjective", "topic"]
template = "Tell me one {adjective} joke about {topic}"

prompt = PromptTemplate(input_variables=input_variables, template=template)

prompt.format(adjective="funny", topic="cars")

'Tell me one funny joke about cars'

In [8]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {
    "adjective": "funny",
    "topic": "cats",
}  # create a dictionary to store the corresponding input to placeholders in prompt template

In [9]:
# chamando via from_templates. Desta forma PromptTemplate nao é instanciado imediatamente.
# A instanciacao acontece em from_template visto se trata de um bound method: <bound method PromptTemplate.from_template of <class 'langchain_core.prompts.prompt.PromptTemplate'>> — callable signature

template = "me conta uma {contar_o_que} sobre {assunto}"
input_variable = {"contar_o_que": "piada", "assunto": "carros"}


prompt = PromptTemplate.from_template(template)
prompt.invoke(input_variable)

StringPromptValue(text='me conta uma piada sobre carros')

## CHAT PROMPT TEMPLATES

Turn chat messages into templates

In [10]:
messages = [("system", "You are a helpful assistant"), ("user", "Tell me a joke about {topic}")]
input_variables = ["topic"]

prompt = ChatPromptTemplate(messages=messages, input_variables=input_variables)
prompt.format(topic="cars")

'System: You are a helpful assistant\nHuman: Tell me a joke about cars'

In [11]:
# another way of calling using from_messages

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), ("user", "Tell me a joke about {topic}")]
)
input_variables = {"topic": "cars"}

prompt.invoke(input_variables)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cars', additional_kwargs={}, response_metadata={})])

## MESSAGE PLACEHOLDER

You can use the MessagesPlaceholder prompt template to add a list of messages in a specific location. In `ChatPromptTemplate.from_messages`, you saw how to format two messages, with each message as a string. But what if you want the user to supply a list of messages that you would slot into a particular spot? You can use `MessagesPlaceholder` for this task.

In [12]:
prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), MessagesPlaceholder("msgs")]
)

input_variables = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

prompt.invoke(input_variables)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the day after Tuesday?', additional_kwargs={}, response_metadata={})])

# OUTPUT PARSERS

Output parsers take the output from an LLM and transform that output to a more suitable format. Parsing the output is very useful when you are using LLMs to generate any form of structured data, or to normalize output from chat models and other LLMs.

In [13]:
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [14]:
joke_query = "Tell me a joke."

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()


In [15]:
format_instructions

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"setup": {"title": "Setup", "description": "question to set up a joke", "type": "string"}, "punchline": {"title": "Punchline", "description": "answer to resolve the joke", "type": "string"}}, "required": ["setup", "punchline"]}\n```'

In [16]:
# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={
        "format_instructions": format_instructions
    },  # Static variables set once when creating the prompt.
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to Llama
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
chain.invoke({"query": joke_query})

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field!'}

## COMMA SEPARATED LIST PARSER

In [17]:
output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()
format_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [18]:
prompt = PromptTemplate(
    input_variables=["subject"],
    template="Answer the user query. {format_instructions} \nList five {subject}",
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | model | output_parser
chain.invoke({"subject": "cars"})
chain = prompt | model | output_parser

['Toyota Corolla',
 'Honda Civic',
 'Ford F-150',
 'Chevrolet Malibu',
 'Nissan Altima']

# DOCUMENT

A `Document` object in `LangChain` contains information about some data. A Document object has the following two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. You can use the metadata to track various details, such as the document ID, the file name, and other details.


In [19]:
doc = Document(
    page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
    metadata={
        "my_document_id": 234234,  # Unique identifier for this document
        "my_document_source": "About Python",  # Source or title information
        "my_document_create_time": 1680013019,  # Unix timestamp for document creation (March 28, 2023)
    },
)

In [20]:
pdf_loader = PyPDFLoader("../../data/LangChain.pdf")
pdf_file = pdf_loader.load()
pdf_file[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-10T11:43:58+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-10T11:43:58+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../../data/LangChain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChainisarapidlyemergingframeworkthatoffersaver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tools for han-\ndling chat models, integrating retrieva

In [21]:
web_loader = WebBaseLoader(web_path="https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()
web_data[0]

Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModel

## TEXT SPLITTERS

One of the most simple examples of making documents better suit your application is to split a long document into smaller chunks that can fit into your model's context window. LangChain has built-in document transformers that ease the process of splitting, combining, filtering, and otherwise manipulating documents.

In [22]:
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(web_data)

Created a chunk of size 1611, which is longer than the specified 200
Created a chunk of size 730, which is longer than the specified 200
Created a chunk of size 835, which is longer than the specified 200


# EMBEDDINGS

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.


In [23]:
embed_param = {"model": "text-embedding-3-small"}

In [24]:
embedding_model = OpenAIEmbeddings(model=embed_param["model"], api_key=parameters["api_key"])

texts = [i.page_content for i in chunks]
embedding_result = embedding_model.embed_documents(texts)


In [25]:
embedding_result[0][:5]

[-0.0194549560546875,
 0.031646728515625,
 0.004749298095703125,
 0.006587982177734375,
 -0.025482177734375]

## VECTOR STORES

One of the most common ways to store and search over unstructured data is to embed the text data and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query.

In [26]:
docs_db = Chroma.from_documents(chunks, embedding_model)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [27]:
type(docs_db)

langchain_community.vectorstores.chroma.Chroma

In [28]:
query = "langchain"
docs = docs_db.similarity_search(query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [29]:
docs[0].page_content

"LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDe

# RETRIEVERS


A retriever is an interface that returns documents using an unstructured query. Retrievers are more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them.

## VECTOR STORE-BACKED RETRIEVERS

Vector store retrievers are retrievers that use a vector store to retrieve documents.

In [30]:
# Embeddings pega um texto unstructured e transforma em vetor.
# o chroma salva esse vetor
# o retriever retorna documentos. Ele pega a query , embeda esta query e consulta no banco de dados, retornando documentos.

In [31]:
# docs_db - onde estao guardados os vetores dos pdf
# embedding_model - onde esta declarado o modelo de embedding

query = "langchain"

retriever = docs_db.as_retriever()

result = retriever.invoke(query)
result[0].page_content[:100]

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: '

## PARENT DOCUMENT RETRIEVER

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data. During retrieval, this retriever first fetches the small chunks, but then looks up the parent IDs for the data and returns those larger documents.


Qual a diference entre o InMemoryStore e o Chroma ? 
<details>

Funcionalmente são dois papéis bem diferentes no mesmo fluxo, e ambos guardam dado — só que dados diferentes, pra propósitos diferentes:

vector_store (Chroma) — o índice de busca
Guarda os pedaços pequenos (child chunks) já transformados em vetor. A função dele é achar — dado uma query, ele faz busca por similaridade semântica entre vetores pra descobrir qual pedacinho de texto é mais relevante. Ele é bom em achar precisão fina, mas um pedaço pequeno geralmente não tem contexto suficiente pra ser útil sozinho como resposta.

store / docstore (InMemoryStore) — o arquivo de referência
Guarda os documentos grandes (parent chunks), como texto puro, indexados por um ID — funciona como uma tabela chave-valor simples, sem nenhuma busca semântica envolvida. A função dele é devolver contexto — uma vez que você já sabe qual pedaço é relevante, você usa o ID desse pedaço pra buscar o documento pai inteiro aqui.

Como os dois se conectam:
Quando o documento original é processado, ele primeiro é dividido em pedaços grandes (pais), e cada pedaço grande é dividido em pedaços pequenos (filhos). Cada filho carrega uma referência pro ID do seu pai. Os filhos (pequenos, embedados) vão pro vector_store. Os pais (grandes, texto puro) vão pro store.

Na hora da busca: a query é comparada contra os vetores dos filhos no vector_store — isso acha o pedaço pequeno mais relevante. Mas em vez de devolver esse pedaço pequeno pra você, o retriever pega o ID do pai daquele filho e busca o pai inteiro no store, devolvendo o documento maior, com mais contexto ao redor do trecho relevante.

Resumindo: o vector_store é onde a pergunta encontra o lugar certo; o store é de onde vem a resposta com contexto suficiente. Um sem o outro não fecha o padrão do ParentDocumentRetriever.

In [32]:
vector_store_ = Chroma(collection_name="demo_1", embedding_function=embedding_model)

child_splitter_ = CharacterTextSplitter(separator="\n", chunk_size=400, chunk_overlap=50)

parent_splitter_ = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

docstore_ = InMemoryStore()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [33]:
retriever = ParentDocumentRetriever(
    vectorstore=vector_store_,
    docstore=docstore_,
    child_splitter=child_splitter_,
    parent_splitter=parent_splitter_,
)

In [34]:
retriever.add_documents(web_data)  # add docs to vectorstore and docstore

Created a chunk of size 1611, which is longer than the specified 400
Created a chunk of size 730, which is longer than the specified 400
Created a chunk of size 835, which is longer than the specified 400


In [35]:
# retrieves and counts the number of parent document IDs stored in the MEMORY document store
len(list(docstore_.yield_keys()))

3

In [36]:
# Next, we verify that the underlying vector store still retrieves the small chunks.
sub_docs = vector_store_.similarity_search(query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [37]:
print(sub_docs[0].page_content)

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDep

In [38]:
# And then retrieve the relevant large chunk.
retrieved_docs = retriever.invoke("Langchain")

# Dentro desse invoke, na ordem:
# Embeda a query "Langchain".
# Faz busca por similaridade no Chroma (vector_store_) contra os embeddings dos child chunks → acha o(s) pedaço(s) pequeno(s) mais relevante(s).
# Pega o ID do documento-pai que está nos metadados desses child chunks.
# Busca no docstore_ (InMemoryStore) o texto completo do pai correspondente a esse ID.
# Devolve o documento pai (não o pedaço pequeno).

In [39]:
print(retrieved_docs[0].page_content[:500])

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangC


# RETRIEVAL QA

> RetrievalQA é Question Answering, não chatbot — cada .invoke() é independente e sem memória.
>> Você pode chamar várias vezes com perguntas diferentes, mas nenhuma sabe da anterior.
Perguntas de acompanhamento ("e ele é gratuito?") não vão entender referências a turnos passados.
>>> Chatbot de verdade precisa de chat_history entre chamadas (ex: ConversationalRetrievalChain) — o que você tem é a base, não isso ainda.

In [40]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Responda à pergunta usando o contexto abaixo.\n\nContexto:\n{context}\n\nPergunta: {question}\nResposta:",
)

In [41]:
# 1 - probe=RetrievalQA pede 2 construtores: combine_documents_chain da classe_1 langchain.chains.combine_documents.base.BaseCombineDocumentsChain'> e retriever, que eu ja conheco.
# 2 - uso a classe_1 como probe e nenhum argumento é requerido, a classe é abstrata, logo  orquestrada automaticamente pelo framework e portanto nao é instanciada direto no codigo.
# 3 - Checo a classe_1 em Siblings para descobrir quem implementa esse contrato?"
# Vejo o modulo stuff  stuff.StuffDocumentsChain como um candidato, classe_2, que pede 2 construtores llm_chain (class LLMChain) e document_variable_name (str)
# 4 - corro probe = LLMChain , pede 2 construtores prompt (BasePromptTemplate) e llm (Runnable — ChatOpenAI ). Visto que eu conheco essas duas classes, já tenho todo o caminho esclarecido.

llm_chain_ = LLMChain(prompt=prompt, llm=model)
question_ = "O que é langchain"

combine_docs_chain = StuffDocumentsChain(llm_chain=llm_chain_, document_variable_name="context")

qa = RetrievalQA(combine_documents_chain=combine_docs_chain, retriever=retriever)

In [42]:
qa.invoke(question_)

{'query': 'O que é langchain',
 'result': 'LangChain é uma estrutura projetada para facilitar a criação de agentes interativos que utilizam modelos de linguagem. Ele fornece uma interface padrão para diferentes modelos de chat, embeddings e outros, permitindo que os desenvolvedores alternem entre modelos com mínimas alterações no código. Através da função `create_agent`, o LangChain oferece um ambiente altamente configurável, onde é possível compor agentes personalizados que atendem a necessidades específicas, integrando modelos, ferramentas, prompts e middleware. Além disso, LangChain é construído sobre LangGraph, promovendo suporte a execução durável, interação humano-na-loop, e outras funcionalidades. O LangSmith, associado ao LangChain, permite rastrear e depurar o comportamento dos agentes e avaliar suas saídas.'}

# MEMORY

Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At a bare minimum, a conversational system should be able to directly access some window of past messages.


## CHAT MESSAGE HISTORY

One of the core utility classes underpinning most (if not all) memory modules is the `ChatMessageHistory` class. This class is a super lightweight wrapper that provides convenience methods for saving `HumanMessages` and `AIMessages`, and then fetching both types of messages.


In [68]:
chat_history = ChatMessageHistory()  # no requireds

# chat_history.clear

In [ ]:
chat_history.add_message(HumanMessage(content="what is the capital of Brazil ?"))
chat_history.add_message(SystemMessage(content="i am a geography teacher "))

# chat_history.add_message(BaseMessage(content="what is the capital of Brazil ?", type="human")) # Funciona para o add_message que adiciona a mensagem ao historico, mas da erro no invoke que nao reconhece BaseMessage.


In [70]:
chat_history.messages

[HumanMessage(content='what is the capital of Brazil ?', additional_kwargs={}, response_metadata={}),
 SystemMessage(content='i am a geography teacher ', additional_kwargs={}, response_metadata={})]

In [71]:
model.invoke(chat_history.messages)

AIMessage(content='The capital of Brazil is Brasília. It was officially inaugurated as the capital in 1960, designed by the architect Oscar Niemeyer and the urban planner Lúcio Costa to promote the development of the interior of the country.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 24, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9709ac0e8b', 'id': 'chatcmpl-ECwOB5Chazc0e2VfdKJha1lvRUH5P', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a002c8-8037-7522-a991-b75b6232dc09-0', usage_metadata={'input_tokens': 24, 'output_tokens': 45, 'total_tokens': 69, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outp

# CONVERSATION BUFFER

Conversation buffer memory allows for the storage of messages, which you use to extract messages to a variable. Consider using conversation buffer memory in a chain, setting `verbose=True` so that the prompt is visible.


In [ ]:
memory = ConversationBufferMemory()

conversation_chain = ConversationChain(

)